In [0]:
# Cell 1 — Load enriched gold Delta layer
# This is our final analysis-ready dataset

delta_enriched = "/Volumes/workspace/default/hmda_raw/delta/enriched_hmda"
df = spark.read.format("delta").load(delta_enriched)

# Register as a Spark SQL temp view
# This lets us write pure SQL in subsequent cells
df.createOrReplaceTempView("hmda")

print(f"Gold layer loaded")
print(f"   Rows    : {df.count():,}")
print(f"   Columns : {len(df.columns)}")
print(f"   Temp view 'hmda' registered for Spark SQL")

Gold layer loaded
   Rows    : 8,007,205
   Columns : 106
   Temp view 'hmda' registered for Spark SQL


In [0]:
# Cell 2 — Research Question 1: Racial Denial Rate Disparities

result_q1 = spark.sql("""
    SELECT
        race_group,
        income_bracket,
        COUNT(*)                                           AS total_applications,
        SUM(is_denied)                                     AS total_denials,
        ROUND(SUM(is_denied) * 100.0 / COUNT(*), 2)       AS denial_rate_pct,
        ROUND(AVG(loan_amount), 0)                         AS avg_loan_amount
    FROM hmda
    WHERE race_group != 'Other/Unknown'
    AND   income_bracket != 'Unknown'
    GROUP BY race_group, income_bracket
    ORDER BY income_bracket, denial_rate_pct DESC
""")

result_q1.show(50, truncate=False)

+---------------+-----------------------+------------------+-------------+---------------+---------------+
|race_group     |income_bracket         |total_applications|total_denials|denial_rate_pct|avg_loan_amount|
+---------------+-----------------------+------------------+-------------+---------------+---------------+
|Native American|High (250k+)           |1767              |533          |30.16          |485702.0       |
|Black          |High (250k+)           |34731             |10079        |29.02          |485897.0       |
|Hispanic       |High (250k+)           |13664             |3946         |28.88          |462559.0       |
|White          |High (250k+)           |533569            |81265        |15.23          |569736.0       |
|Asian          |High (250k+)           |128683            |18156        |14.11          |737326.0       |
|Hispanic       |Low (<50k)             |39258             |26534        |67.59          |118085.0       |
|Native American|Low (<50k)          

In [0]:
# Cell 3 — Compute explicit disparity gap vs White applicants
# This makes the finding crystal clear

disparity = spark.sql("""
    WITH base AS (
        SELECT
            race_group,
            income_bracket,
            ROUND(SUM(is_denied) * 100.0 / COUNT(*), 2) AS denial_rate
        FROM hmda
        WHERE race_group != 'Other/Unknown'
        AND   income_bracket != 'Unknown'
        GROUP BY race_group, income_bracket
    ),
    white_rates AS (
        SELECT income_bracket, denial_rate AS white_denial_rate
        FROM base
        WHERE race_group = 'White'
    )
    SELECT
        b.race_group,
        b.income_bracket,
        b.denial_rate,
        w.white_denial_rate,
        ROUND(b.denial_rate - w.white_denial_rate, 2)      AS gap_vs_white,
        ROUND(b.denial_rate / w.white_denial_rate, 2)       AS ratio_vs_white
    FROM base b
    JOIN white_rates w ON b.income_bracket = w.income_bracket
    WHERE b.race_group != 'White'
    ORDER BY gap_vs_white DESC
""")

disparity.show(30, truncate=False)

+---------------+-----------------------+-----------+-----------------+------------+--------------+
|race_group     |income_bracket         |denial_rate|white_denial_rate|gap_vs_white|ratio_vs_white|
+---------------+-----------------------+-----------+-----------------+------------+--------------+
|Hispanic       |Low (<50k)             |67.59      |45.82            |21.77       |1.48          |
|Native American|Low (<50k)             |65.04      |45.82            |19.22       |1.42          |
|Native American|High (250k+)           |30.16      |15.23            |14.93       |1.98          |
|Hispanic       |Moderate (50-100k)     |39.84      |25.24            |14.60       |1.58          |
|Black          |Low (<50k)             |59.87      |45.82            |14.05       |1.31          |
|Black          |High (250k+)           |29.02      |15.23            |13.79       |1.91          |
|Hispanic       |High (250k+)           |28.88      |15.23            |13.65       |1.90          |


In [0]:
# Cell 4 — Research Question 2
# Are minority borrowers charged higher rate spreads for equivalent loans?
# We look at ORIGINATED loans only (action_taken = 1)
# and control for loan_type and lien_status

result_q2 = spark.sql("""
    SELECT
        race_group,
        loan_type,
        COUNT(*)                                        AS total_loans,
        ROUND(AVG(rate_spread), 4)                      AS avg_rate_spread,
        ROUND(AVG(interest_rate), 4)                    AS avg_interest_rate,
        ROUND(AVG(loan_amount), 0)                      AS avg_loan_amount,
        ROUND(AVG(income), 0)                           AS avg_income
    FROM hmda
    WHERE action_taken    = 1
    AND   race_group     != 'Other/Unknown'
    AND   rate_spread     IS NOT NULL
    AND   interest_rate   IS NOT NULL
    AND   lien_status     = 1
    GROUP BY race_group, loan_type
    ORDER BY loan_type, avg_rate_spread DESC
""")

result_q2.show(30, truncate=False)

+---------------+---------+-----------+---------------+-----------------+---------------+----------+
|race_group     |loan_type|total_loans|avg_rate_spread|avg_interest_rate|avg_loan_amount|avg_income|
+---------------+---------+-----------+---------------+-----------------+---------------+----------+
|Hispanic       |1        |68179      |0.6059         |6.9791           |314072.0       |132.0     |
|Native American|1        |6182       |0.5416         |6.9989           |293790.0       |138.0     |
|Black          |1        |157212     |0.4016         |6.8116           |297361.0       |130.0     |
|White          |1        |1808020    |0.354          |6.8512           |347277.0       |168.0     |
|Asian          |1        |303506     |0.0776         |6.5262           |543144.0       |212.0     |
|White          |2        |444990     |0.5087         |6.3542           |298302.0       |101.0     |
|Native American|2        |2281       |0.4612         |6.3255           |288336.0       |99

In [0]:
# Cell 5 — Explicit rate spread gap vs White borrowers
# Controlling for loan_type (apples to apples comparison)

spread_disparity = spark.sql("""
    WITH base AS (
        SELECT
            race_group,
            loan_type,
            ROUND(AVG(rate_spread), 4)    AS avg_spread,
            ROUND(AVG(interest_rate), 4)  AS avg_rate,
            COUNT(*)                       AS n
        FROM hmda
        WHERE action_taken  = 1
        AND   race_group   != 'Other/Unknown'
        AND   rate_spread   IS NOT NULL
        AND   interest_rate IS NOT NULL
        AND   lien_status   = 1
        AND   loan_type     = 1
        GROUP BY race_group, loan_type
    ),
    white_base AS (
        SELECT avg_spread AS white_spread, avg_rate AS white_rate
        FROM base WHERE race_group = 'White'
    )
    SELECT
        b.race_group,
        b.n                                                     AS total_loans,
        b.avg_spread,
        w.white_spread,
        ROUND(b.avg_spread - w.white_spread, 4)                 AS spread_gap,
        b.avg_rate,
        w.white_rate,
        ROUND(b.avg_rate - w.white_rate, 4)                     AS rate_gap
    FROM base b
    CROSS JOIN white_base w
    WHERE b.race_group != 'White'
    ORDER BY spread_gap DESC
""")

print("=== Rate Spread Disparity vs White (Conventional Loans) ===")
spread_disparity.show(truncate=False)

# Dollar impact calculation
print("=== Dollar impact of rate spread gap ===")
spark.sql("""
    WITH gaps AS (
        SELECT
            race_group,
            ROUND(AVG(rate_spread) - (
                SELECT AVG(rate_spread) 
                FROM hmda 
                WHERE race_group = 'White' 
                AND action_taken = 1 
                AND loan_type = 1
                AND rate_spread IS NOT NULL
            ), 4) AS spread_gap,
            ROUND(AVG(loan_amount), 0) AS avg_loan
        FROM hmda
        WHERE action_taken  = 1
        AND   race_group   != 'Other/Unknown'
        AND   race_group   != 'White'
        AND   rate_spread   IS NOT NULL
        AND   loan_type     = 1
        GROUP BY race_group
    )
    SELECT
        race_group,
        spread_gap,
        avg_loan,
        -- Extra interest paid over 30 year mortgage
        ROUND(avg_loan * (spread_gap/100) * 30, 0) AS extra_interest_30yr
    FROM gaps
    ORDER BY extra_interest_30yr DESC
""").show(truncate=False)

=== Rate Spread Disparity vs White (Conventional Loans) ===
+---------------+-----------+----------+------------+----------+--------+----------+--------+
|race_group     |total_loans|avg_spread|white_spread|spread_gap|avg_rate|white_rate|rate_gap|
+---------------+-----------+----------+------------+----------+--------+----------+--------+
|Hispanic       |68179      |0.6059    |0.354       |0.2519    |6.9791  |6.8512    |0.1279  |
|Native American|6182       |0.5416    |0.354       |0.1876    |6.9989  |6.8512    |0.1477  |
|Black          |157212     |0.4016    |0.354       |0.0476    |6.8116  |6.8512    |-0.0396 |
|Asian          |303506     |0.0776    |0.354       |-0.2764   |6.5262  |6.8512    |-0.325  |
+---------------+-----------+----------+------------+----------+--------+----------+--------+

=== Dollar impact of rate spread gap ===
+---------------+----------+--------+-------------------+
|race_group     |spread_gap|avg_loan|extra_interest_30yr|
+---------------+----------+--

In [0]:
# Cell 6 — Research Question 3
# Which lenders best/worst serve minority and LMI communities?
# Among top 50 lenders by origination volume

result_q3 = spark.sql("""
    WITH lender_stats AS (
        SELECT
            lei,
            COUNT(*)                                           AS total_applications,
            SUM(CASE WHEN action_taken = 1 THEN 1 ELSE 0 END) AS total_originated,

            -- Minority serving metrics
            ROUND(SUM(is_minority) * 100.0 / COUNT(*), 2)     AS minority_app_pct,
            ROUND(SUM(CASE WHEN action_taken = 1 
                AND is_minority = 1 THEN 1 ELSE 0 END) * 100.0 
                / NULLIF(SUM(CASE WHEN action_taken = 1 
                THEN 1 ELSE 0 END), 0), 2)                    AS minority_originated_pct,

            -- LMI tract serving metrics
            ROUND(SUM(CASE WHEN is_lmi_tract = 1 
                THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)     AS lmi_tract_app_pct,
            ROUND(SUM(CASE WHEN action_taken = 1 
                AND is_lmi_tract = 1 THEN 1 ELSE 0 END) * 100.0
                / NULLIF(SUM(CASE WHEN action_taken = 1 
                THEN 1 ELSE 0 END), 0), 2)                    AS lmi_originated_pct,

            -- Overall denial rate
            ROUND(SUM(is_denied) * 100.0 / COUNT(*), 2)       AS overall_denial_rate,

            -- Minority denial rate
            ROUND(SUM(CASE WHEN is_minority = 1 
                AND is_denied = 1 THEN 1 ELSE 0 END) * 100.0
                / NULLIF(SUM(is_minority), 0), 2)             AS minority_denial_rate,

            -- White denial rate
            ROUND(SUM(CASE WHEN race_group = 'White' 
                AND is_denied = 1 THEN 1 ELSE 0 END) * 100.0
                / NULLIF(SUM(CASE WHEN race_group = 'White' 
                THEN 1 ELSE 0 END), 0), 2)                    AS white_denial_rate

        FROM hmda
        GROUP BY lei
    ),
    top_50 AS (
        SELECT * FROM lender_stats
        ORDER BY total_originated DESC
        LIMIT 50
    )
    SELECT
        lei,
        total_applications,
        total_originated,
        minority_app_pct,
        minority_originated_pct,
        lmi_tract_app_pct,
        lmi_originated_pct,
        overall_denial_rate,
        minority_denial_rate,
        white_denial_rate,
        ROUND(minority_denial_rate - white_denial_rate, 2) AS denial_gap
    FROM top_50
    ORDER BY minority_originated_pct DESC
""")

print("=== Top 50 Lenders — Best Minority Serving (Top 10) ===")
result_q3.show(10, truncate=False)

print("=== Top 50 Lenders — Worst Minority Serving (Bottom 10) ===")
result_q3.orderBy("minority_originated_pct").show(10, truncate=False)

print("=== Top 50 Lenders — Largest Denial Gap (CRA Risk) ===")
result_q3.orderBy("denial_gap", ascending=False).show(10, truncate=False)

=== Top 50 Lenders — Best Minority Serving (Top 10) ===
+--------------------+------------------+----------------+----------------+-----------------------+-----------------+------------------+-------------------+--------------------+-----------------+----------+
|lei                 |total_applications|total_originated|minority_app_pct|minority_originated_pct|lmi_tract_app_pct|lmi_originated_pct|overall_denial_rate|minority_denial_rate|white_denial_rate|denial_gap|
+--------------------+------------------+----------------+----------------+-----------------------+-----------------+------------------+-------------------+--------------------+-----------------+----------+
|254900ZFWS2106HWPH46|23626             |22089           |50.10           |49.41                  |26.61            |26.20             |5.84               |7.10                |5.39             |1.71      |
|5493001SXWZ4OFP8Z903|99961             |88678           |41.61           |39.85                  |14.73            

In [0]:
# Cell 7 — Look up lender names using LEI
# LEI = Legal Entity Identifier — a global standard for financial institutions
# We can decode them using the GLEIF API

import requests

# Top CRA risk lenders by denial gap
risk_leis = [
    "E57ODZWZ7FF32TWEFA76",
    "WWB2V0FCW3A0EE3ZJN75",
    "AD6GFRVSDT01YPT1CS68",
    "5493008CPTDVOS570626",
    "5493006VAGP3GQ8FJT49",   # zero minority apps
    "254900ZFWS2106HWPH46",   # best minority serving
    "2WHM8VNJH63UN14OL754",
    "DRMSV1Q0EKMEXLAU1P80",
    "6BYL5QZYBDK8S7L73M02",
    "N8T7HW55LK5D2ORCKP39"
]

print("=== Lender Name Lookup via GLEIF API ===")
print(f"{'LEI':<25} {'Institution Name':<50}")
print("-" * 75)

for lei in risk_leis:
    try:
        url = f"https://api.gleif.org/api/v1/lei-records/{lei}"
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            name = r.json()["data"]["attributes"]["entity"]["legalName"]["name"]
        else:
            name = "Not found"
    except Exception as e:
        name = f"Error: {e}"
    print(f"{lei:<25} {name:<50}")

=== Lender Name Lookup via GLEIF API ===
LEI                       Institution Name                                  
---------------------------------------------------------------------------
E57ODZWZ7FF32TWEFA76      Citibank, National Association                    
WWB2V0FCW3A0EE3ZJN75      Manufacturers and Traders Trust Company           
AD6GFRVSDT01YPT1CS68      PNC Bank, National Association                    
5493008CPTDVOS570626      Third Federal Savings And Loan Association Of Cleveland
5493006VAGP3GQ8FJT49      Kiavi Funding, Inc.                               
254900ZFWS2106HWPH46      PARAMOUNT RESIDENTIAL MORTGAGE GROUP, INC.        
2WHM8VNJH63UN14OL754      The Huntington National Bank                      
DRMSV1Q0EKMEXLAU1P80      Citizens Bank, National Association               
6BYL5QZYBDK8S7L73M02      U.S. Bank National Association                    
N8T7HW55LK5D2ORCKP39      First National Bank of Pennsylvania               


In [0]:
# Cell 8 — Save all analysis results as Delta tables

results_base = "/Volumes/workspace/default/hmda_raw/delta/results"

# Save Q1 results
result_q1.write.format("delta").mode("overwrite").save(f"{results_base}/q1_denial_rates")

# Save Q1 disparity
disparity.write.format("delta").mode("overwrite").save(f"{results_base}/q1_disparity")

# Save Q2 results
result_q2.write.format("delta").mode("overwrite").save(f"{results_base}/q2_rate_spread")

# Save Q2 spread disparity
spread_disparity.write.format("delta").mode("overwrite").save(f"{results_base}/q2_spread_disparity")

# Save Q3 results
result_q3.write.format("delta").mode("overwrite").save(f"{results_base}/q3_lender_fairness")

print("All analysis results saved to Delta")
print(f"\nResults saved:")
print(f"  {results_base}/q1_denial_rates")
print(f"  {results_base}/q1_disparity")
print(f"  {results_base}/q2_rate_spread")
print(f"  {results_base}/q2_spread_disparity")
print(f"  {results_base}/q3_lender_fairness")

All analysis results saved to Delta

Results saved:
  /Volumes/workspace/default/hmda_raw/delta/results/q1_denial_rates
  /Volumes/workspace/default/hmda_raw/delta/results/q1_disparity
  /Volumes/workspace/default/hmda_raw/delta/results/q2_rate_spread
  /Volumes/workspace/default/hmda_raw/delta/results/q2_spread_disparity
  /Volumes/workspace/default/hmda_raw/delta/results/q3_lender_fairness


In [0]:
# ── Q4: Mortgage Credit Deserts ──────────────────────────
# Identifying census tracts that are geographically underserved
# Low application volume + High denial rates for LMI applicants

q4_credit_deserts = spark.sql("""
    WITH tract_stats AS (
        SELECT
            state_code,
            county_code,
            census_tract,
            tract_income_category,
            tract_to_msa_income_percentage,
            tract_minority_population_percent,

            COUNT(*)                                              AS total_applications,
            SUM(is_denied)                                        AS total_denials,
            ROUND(SUM(is_denied) * 100.0 / COUNT(*), 2)          AS denial_rate_pct,

            -- LMI applicant denial rate
            ROUND(SUM(CASE WHEN is_lmi_tract = 1 
                AND is_denied = 1 THEN 1 ELSE 0 END) * 100.0
                / NULLIF(SUM(CASE WHEN is_lmi_tract = 1 
                THEN 1 ELSE 0 END), 0), 2)                       AS lmi_denial_rate,

            -- Minority applicant count
            SUM(is_minority)                                      AS minority_applications,
            ROUND(SUM(is_minority) * 100.0 / COUNT(*), 2)        AS minority_app_pct

        FROM hmda
        WHERE census_tract IS NOT NULL
        AND   county_code  IS NOT NULL
        GROUP BY
            state_code,
            county_code,
            census_tract,
            tract_income_category,
            tract_to_msa_income_percentage,
            tract_minority_population_percent
    )
    SELECT *,
        -- Credit desert score
        -- High score = underserved
        -- Low applications + High denial rate = credit desert
        ROUND(
            (denial_rate_pct * 0.5) +
            (lmi_denial_rate * 0.3) +
            (minority_app_pct * 0.2),
        2) AS desert_score

    FROM tract_stats
    WHERE total_applications >= 10    -- minimum threshold for reliability
    AND   tract_income_category IN ('Low Income', 'Moderate Income')
    ORDER BY desert_score DESC
""")

print("=== Top 20 Mortgage Credit Deserts ===")
q4_credit_deserts.show(20, truncate=False)
print(f"Total LMI tracts analyzed: {q4_credit_deserts.count():,}")

=== Top 20 Mortgage Credit Deserts ===
+----------+-----------+------------+---------------------+------------------------------+---------------------------------+------------------+-------------+---------------+---------------+---------------------+----------------+------------+
|state_code|county_code|census_tract|tract_income_category|tract_to_msa_income_percentage|tract_minority_population_percent|total_applications|total_denials|denial_rate_pct|lmi_denial_rate|minority_applications|minority_app_pct|desert_score|
+----------+-----------+------------+---------------------+------------------------------+---------------------------------+------------------+-------------+---------------+---------------+---------------------+----------------+------------+
|FL        |12086      |12086010015 |Moderate Income      |70.0                          |95.28                            |12                |12           |100.00         |100.00         |11                   |91.67           |98.33  

In [0]:
# County level aggregation for visualization
q4_county = spark.sql("""
    SELECT
        state_code,
        county_code,
        COUNT(DISTINCT census_tract)                          AS total_tracts,
        COUNT(*)                                              AS total_applications,
        ROUND(SUM(is_denied) * 100.0 / COUNT(*), 2)          AS denial_rate_pct,
        ROUND(AVG(tract_minority_population_percent), 2)      AS avg_minority_pct,
        ROUND(AVG(tract_to_msa_income_percentage), 2)         AS avg_tract_income_pct,
        SUM(CASE WHEN is_lmi_tract = 1 THEN 1 ELSE 0 END)    AS lmi_applications,
        ROUND(SUM(CASE WHEN is_lmi_tract = 1 
            AND is_denied = 1 THEN 1 ELSE 0 END) * 100.0
            / NULLIF(SUM(CASE WHEN is_lmi_tract = 1 
            THEN 1 ELSE 0 END), 0), 2)                       AS lmi_denial_rate
    FROM hmda
    WHERE county_code IS NOT NULL
    GROUP BY state_code, county_code
    ORDER BY lmi_denial_rate DESC
""")

print("=== Top 20 Counties by LMI Denial Rate ===")
q4_county.show(20, truncate=False)

# Save results
q4_credit_deserts.write.format("delta").mode("overwrite") \
    .save("/Volumes/workspace/default/hmda_raw/delta/results/q4_credit_deserts")
q4_county.write.format("delta").mode("overwrite") \
    .save("/Volumes/workspace/default/hmda_raw/delta/results/q4_county")

print("Q4 results saved")

=== Top 20 Counties by LMI Denial Rate ===
+----------+-----------+------------+------------------+---------------+----------------+--------------------+----------------+---------------+
|state_code|county_code|total_tracts|total_applications|denial_rate_pct|avg_minority_pct|avg_tract_income_pct|lmi_applications|lmi_denial_rate|
+----------+-----------+------------+------------------+---------------+----------------+--------------------+----------------+---------------+
|NC        |37139      |11          |2927              |28.05          |45.8            |126.51              |198             |71.21          |
|TX        |48307      |4           |208               |38.94          |35.8            |94.88               |31              |67.74          |
|TX        |48219      |8           |1230              |48.05          |44.25           |86.69               |246             |63.01          |
|IL        |17159      |5           |264               |36.36          |6.65            |106.

In [0]:
# ── Q5: Fed Rate Cycle Impact Across Demographic Groups ──────
# 2023 = peak rates (7.5-8%)
# 2024 = Fed cut rates 3 times (-100 basis points)
# Did rate relief benefit all groups equally?

q5_rate_cycle = spark.sql("""
    WITH yearly_stats AS (
        SELECT
            activity_year,
            race_group,
            income_bracket,

            COUNT(*)                                            AS total_applications,
            SUM(is_denied)                                      AS total_denials,
            ROUND(SUM(is_denied) * 100.0 / COUNT(*), 2)        AS denial_rate_pct,

            ROUND(AVG(loan_amount), 0)                          AS avg_loan_amount,
            ROUND(AVG(interest_rate), 4)                        AS avg_interest_rate,
            ROUND(AVG(rate_spread), 4)                          AS avg_rate_spread,

            -- Origination rate (approved AND funded)
            ROUND(SUM(CASE WHEN action_taken = 1 
                THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)      AS origination_rate

        FROM hmda
        WHERE race_group != 'Other/Unknown'
        AND   income_bracket != 'Unknown'
        GROUP BY activity_year, race_group, income_bracket
    )
    SELECT * FROM yearly_stats
    ORDER BY race_group, income_bracket, activity_year
""")

print("=== Year over Year Stats by Race and Income ===")
q5_rate_cycle.show(50, truncate=False)

=== Year over Year Stats by Race and Income ===
+-------------+---------------+-----------------------+------------------+-------------+---------------+---------------+-----------------+---------------+----------------+
|activity_year|race_group     |income_bracket         |total_applications|total_denials|denial_rate_pct|avg_loan_amount|avg_interest_rate|avg_rate_spread|origination_rate|
+-------------+---------------+-----------------------+------------------+-------------+---------------+---------------+-----------------+---------------+----------------+
|2023         |Asian          |High (250k+)           |55702             |8323         |14.94          |713408.0       |6.7684           |0.202          |80.49           |
|2024         |Asian          |High (250k+)           |72981             |9833         |13.47          |755581.0       |6.8771           |0.2353         |82.11           |
|2023         |Asian          |Low (<50k)             |14504             |7856         |54.1

In [0]:
# Cell — Compute explicit Year over Year change
q5_yoy = spark.sql("""
    WITH yearly AS (
        SELECT
            activity_year,
            race_group,
            COUNT(*)                                        AS total_apps,
            ROUND(SUM(is_denied) * 100.0 / COUNT(*), 2)   AS denial_rate,
            ROUND(AVG(interest_rate), 4)                   AS avg_rate,
            ROUND(AVG(loan_amount), 0)                     AS avg_loan,
            ROUND(SUM(CASE WHEN action_taken = 1 
                THEN 1 ELSE 0 END) * 100.0 
                / COUNT(*), 2)                             AS origination_rate
        FROM hmda
        WHERE race_group != 'Other/Unknown'
        GROUP BY activity_year, race_group
    ),
    y2023 AS (SELECT * FROM yearly WHERE activity_year = 2023),
    y2024 AS (SELECT * FROM yearly WHERE activity_year = 2024)
    
    SELECT
        y2023.race_group,
        y2023.total_apps                                   AS apps_2023,
        y2024.total_apps                                   AS apps_2024,
        ROUND((y2024.total_apps - y2023.total_apps) 
            * 100.0 / y2023.total_apps, 2)                AS app_volume_change_pct,
        y2023.denial_rate                                  AS denial_2023,
        y2024.denial_rate                                  AS denial_2024,
        ROUND(y2024.denial_rate - y2023.denial_rate, 2)   AS denial_rate_change,
        y2023.avg_rate                                     AS rate_2023,
        y2024.avg_rate                                     AS rate_2024,
        ROUND(y2024.avg_rate - y2023.avg_rate, 4)         AS interest_rate_change,
        y2023.origination_rate                             AS orig_rate_2023,
        y2024.origination_rate                             AS orig_rate_2024,
        ROUND(y2024.origination_rate 
            - y2023.origination_rate, 2)                  AS orig_rate_change
    FROM y2023
    JOIN y2024 ON y2023.race_group = y2024.race_group
    ORDER BY denial_rate_change ASC
""")

print("=== Year over Year Change by Race (2023 → 2024) ===")
print("Negative denial_rate_change = IMPROVEMENT")
q5_yoy.show(truncate=False)

# Save
q5_rate_cycle.write.format("delta").mode("overwrite") \
    .save("/Volumes/workspace/default/hmda_raw/delta/results/q5_rate_cycle")
q5_yoy.write.format("delta").mode("overwrite") \
    .save("/Volumes/workspace/default/hmda_raw/delta/results/q5_yoy")

print("Q5 results saved")

=== Year over Year Change by Race (2023 → 2024) ===
Negative denial_rate_change = IMPROVEMENT
+---------------+---------+---------+---------------------+-----------+-----------+------------------+---------+---------+--------------------+--------------+--------------+----------------+
|race_group     |apps_2023|apps_2024|app_volume_change_pct|denial_2023|denial_2024|denial_rate_change|rate_2023|rate_2024|interest_rate_change|orig_rate_2023|orig_rate_2024|orig_rate_change|
+---------------+---------+---------+---------------------+-----------+-----------+------------------+---------+---------+--------------------+--------------+--------------+----------------+
|Native American|13028    |13241    |1.63                 |41.89      |37.35      |-4.54             |7.2382   |7.303    |0.0648              |54.63         |58.12         |3.49            |
|Hispanic       |132549   |146983   |10.89                |41.21      |37.02      |-4.19             |7.191    |7.1985   |0.0075              